In [1]:
!nvidia-smi

Wed Sep 16 13:28:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import files
uploaded = files.upload()

Saving agentlab_qwen_lora_7b.zip to agentlab_qwen_lora_7b.zip


In [5]:
import zipfile, os, glob

zip_files = glob.glob("/content/*.zip")
print("Zips found:", zip_files)

for z in zip_files:
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall("/content/")
    print(f"Extracted: {z}")

print("\nSearching for adapter_config.json...")
found = []
for root, dirs, files in os.walk("/content"):
    if "adapter_config.json" in files:
        found.append(root)
        print("FOUND:", root)

# Save the path for later
if found:
    with open("/content/adapter_path.txt", "w") as f:
        f.write(found[0])
    print(f"\n✅ Adapter path saved: {found[0]}")
else:
    print("\n❌ No adapter_config.json found. Check your zip.")

Zips found: ['/content/agentlab_qwen_lora_7b.zip']
Extracted: /content/agentlab_qwen_lora_7b.zip

Searching for adapter_config.json...
FOUND: /content/content/agentlab_qwen_lora_7b
FOUND: /content/content/agentlab_qwen_lora_7b/checkpoint-15
FOUND: /content/content/agentlab_qwen_lora_7b/checkpoint-10
FOUND: /content/content/agentlab_qwen_lora_7b/checkpoint-5

✅ Adapter path saved: /content/content/agentlab_qwen_lora_7b


In [6]:
import json, os

with open("/content/adapter_path.txt") as f:
    ADAPTER_PATH = f.read().strip()

print("Adapter path:", ADAPTER_PATH)
print("Files:", os.listdir(ADAPTER_PATH))

with open(f"{ADAPTER_PATH}/adapter_config.json") as f:
    cfg = json.load(f)

print("\n--- adapter_config.json key fields ---")
print("base_model_name_or_path:", cfg.get("base_model_name_or_path"))
print("r (LoRA rank):", cfg.get("r"))
print("target_modules:", cfg.get("target_modules"))

size = os.path.getsize(f"{ADAPTER_PATH}/adapter_model.safetensors")
print(f"\nadapter_model.safetensors size: {size / 1e6:.1f} MB")

Adapter path: /content/content/agentlab_qwen_lora_7b
Files: ['adapter_config.json', 'tokenizer.json', 'merges.txt', 'training_args.bin', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoint-15', 'checkpoint-10', 'checkpoint-5', 'special_tokens_map.json', 'vocab.json', 'README.md', 'tokenizer_config.json', 'added_tokens.json']

--- adapter_config.json key fields ---
base_model_name_or_path: Qwen/Qwen2.5-7B-Instruct
r (LoRA rank): 16
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']

adapter_model.safetensors size: 40.4 MB


In [7]:
!pip install -q transformers peft accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio requests

In [8]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

with open("/content/adapter_path.txt") as f:
    ADAPTER_PATH = f.read().strip()

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model in 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)

print("Applying V1 LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print(f"\n✅ Model ready. Adapter: {ADAPTER_PATH}")

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Applying V1 LoRA adapter...

✅ Model ready. Adapter: /content/content/agentlab_qwen_lora_7b


In [9]:
!pip install -q --upgrade bitsandbytes

In [10]:
import re, json
from fastapi import FastAPI, Request
import uvicorn, threading, nest_asyncio
from pyngrok import ngrok

app = FastAPI()

TOOL_CALL_CLOSED = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
TOOL_CALL_OPEN   = re.compile(r"<tool_call>\s*(\{.*?\})", re.DOTALL)

def parse_model_output(raw_text: str) -> dict:
    matches = TOOL_CALL_CLOSED.findall(raw_text)
    if matches:
        calls = []
        for m in matches:
            try:
                p = json.loads(m)
                if p.get("name"):
                    calls.append({"name": p["name"], "arguments": p.get("arguments", {})})
            except json.JSONDecodeError:
                continue
        if calls:
            return {"tool_calls": calls, "text": None}

    m = TOOL_CALL_OPEN.search(raw_text)
    if m:
        try:
            p = json.loads(m.group(1))
            if p.get("name"):
                return {"tool_calls": [{"name": p["name"], "arguments": p.get("arguments", {})}], "text": None}
        except json.JSONDecodeError:
            pass

    return {"tool_calls": [], "text": raw_text.strip()}


@app.post("/chat")
async def chat(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    tools = body.get("tools", [])

    prompt = tokenizer.apply_chat_template(
        messages, tools=tools if tools else None,
        add_generation_prompt=True, tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("=" * 50)
    print("[RAW]", raw[:800])
    parsed = parse_model_output(raw)
    print("[PARSED]", parsed)
    print("=" * 50)

    return parsed


@app.get("/health")
async def health():
    return {"status": "ok", "adapter": ADAPTER_PATH, "parser": "v2_two_pass"}

print("✅ Server object defined")

✅ Server object defined


In [11]:
NGROK_AUTH_TOKEN = "30x34MIoplFkIzcMdvm1Ti8cB7T_3Dp9L4kX2JozPrZQZMHvV"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

public_url = ngrok.connect(8000)
E4_URL = public_url.public_url

print("\n" + "=" * 60)
print("E4 SERVER URL:")
print(E4_URL)
print("=" * 60)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

with open("/content/e4_url.txt", "w") as f:
    f.write(E4_URL)

print("\n✅ Server started. URL saved.")


E4 SERVER URL:
https://471b-34-142-230-136.ngrok-free.app

✅ Server started. URL saved.


In [12]:
import requests

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

# Health check
h = requests.get(f"{E4_URL}/health", timeout=15)
print("Health:", h.status_code, h.json())

# Test with one small prompt
test = requests.post(
    f"{E4_URL}/chat",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful AI agent with access to tools: calculator, word_count, web_search, and get_weather."},
            {"role": "user", "content": "What is 84 times 19?"},
        ],
        "tools": [{
            "type": "function",
            "function": {
                "name": "calculator",
                "description": "Evaluate arithmetic",
                "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
            },
        }],
    },
    timeout=120,
)
print("\nTest:", test.status_code)
print(test.json())

INFO:     34.142.230.136:0 - "GET /health HTTP/1.1" 200 OK
Health: 200 {'status': 'ok', 'adapter': '/content/content/agentlab_qwen_lora_7b', 'parser': 'v2_two_pass'}
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "84 * 19"}}
</tool_call>
[PARSED] {'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '84 * 19'}}], 'text': None}
INFO:     34.142.230.136:0 - "POST /chat HTTP/1.1" 200 OK

Test: 200
{'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '84 * 19'}}], 'text': None}


In [14]:
from google.colab import files
uploaded = files.upload()

Saving tasks.py to tasks.py


In [15]:
from google.colab import files
uploaded = files.upload()

Saving harness.py to harness.py


In [16]:
import os, shutil
os.makedirs("/content/eval", exist_ok=True)
for f in ["tasks.py", "harness.py"]:
    if os.path.exists(f"/content/{f}"):
        shutil.move(f"/content/{f}", f"/content/eval/{f}")
print("Files in /content/eval:", os.listdir("/content/eval"))

Files in /content/eval: ['tasks.py', 'harness.py']


In [18]:
# Setup + unzip + load model + start server — all in one cell
import zipfile, os, glob, json, re, threading, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from fastapi import FastAPI, Request
import uvicorn, nest_asyncio, requests
from pyngrok import ngrok

# 1) Unzip adapter (skip if already present)
if not glob.glob("/content/**/adapter_config.json", recursive=True):
    from google.colab import files
    uploaded = files.upload()
for z in glob.glob("/content/*.zip"):
    with zipfile.ZipFile(z) as zf: zf.extractall("/content/")
ADAPTER_PATH = [r for r,_,fs in os.walk("/content") if "adapter_config.json" in fs][0]
print("Adapter:", ADAPTER_PATH)

# 2) Load model + V1 adapter
BASE = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"),
    device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH).eval()
print("✅ model loaded")

# 3) Parser + FastAPI
CLOSED = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
OPEN   = re.compile(r"<tool_call>\s*(\{.*?\})", re.DOTALL)
def parse(raw):
    ms = CLOSED.findall(raw)
    if ms:
        calls = []
        for m in ms:
            try:
                p = json.loads(m)
                if p.get("name"): calls.append({"name": p["name"], "arguments": p.get("arguments", {})})
            except: continue
        if calls: return {"tool_calls": calls, "text": None}
    m = OPEN.search(raw)
    if m:
        try:
            p = json.loads(m.group(1))
            if p.get("name"): return {"tool_calls":[{"name":p["name"],"arguments":p.get("arguments",{})}],"text":None}
        except: pass
    return {"tool_calls": [], "text": raw.strip()}

app = FastAPI()
@app.post("/chat")
async def chat(request: Request):
    body = await request.json()
    prompt = tokenizer.apply_chat_template(body.get("messages",[]),
        tools=body.get("tools") or None, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse(raw)

@app.get("/health")
async def health(): return {"status": "ok"}

ngrok.kill()
ngrok.set_auth_token("30x34MIoplFkIzcMdvm1Ti8cB7T_3Dp9L4kX2JozPrZQZMHvV")
nest_asyncio.apply()
E4_URL = ngrok.connect(8000).public_url
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
with open("/content/e4_url.txt","w") as f: f.write(E4_URL)
time.sleep(5)
print("Health:", requests.get(f"{E4_URL}/health", timeout=15).text)
print("URL:", E4_URL)

Adapter: /content/content/agentlab_qwen_lora_7b


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ model loaded


INFO:     Started server process [1023]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     34.142.230.136:0 - "GET /health HTTP/1.1" 200 OK
Health: {"status":"ok","adapter":"/content/content/agentlab_qwen_lora_7b","parser":"v2_two_pass"}
URL: https://ae06-34-142-230-136.ngrok-free.app


In [19]:
import os, shutil
os.makedirs("/content/eval", exist_ok=True)
for f in ["tasks.py", "harness.py"]:
    if os.path.exists(f"/content/{f}"):
        shutil.move(f"/content/{f}", f"/content/eval/{f}")
print("Files in /content/eval:", os.listdir("/content/eval"))

Files in /content/eval: ['tasks.py', '__init__.py', 'harness.py']


In [20]:
import os, shutil, sys, types
os.makedirs("/content/eval", exist_ok=True)
open("/content/eval/__init__.py","a").close()
for f in list(os.listdir("/content")):
    if f.startswith("tasks") and f.endswith(".py"):
        shutil.move(f"/content/{f}", "/content/eval/tasks.py")
    if f.startswith("harness") and f.endswith(".py"):
        shutil.move(f"/content/{f}", "/content/eval/harness.py")
print("eval/:", os.listdir("/content/eval"))

fake = types.ModuleType("agent")
class Agent: pass
class RunTrace:
    def __init__(self, *a, **k):
        self.tool_calls=k.get("tool_calls",[]); self.tool_results=k.get("tool_results",[])
        self.steps=k.get("steps",0); self.final_answer=k.get("final_answer","")
        self.hit_max_steps=k.get("hit_max_steps",False)
class ToolCall:
    def __init__(self, name=None, arguments=None, **k):
        self.name, self.arguments = name, arguments or {}
class ToolResult:
    def __init__(self, name=None, output=None, error=None, **k):
        self.name, self.output, self.error = name, output, error
fake.Agent, fake.RunTrace, fake.ToolCall, fake.ToolResult = Agent, RunTrace, ToolCall, ToolResult
sys.modules["agent"] = fake
print("✅ agent stub ready")

eval/: ['tasks.py', '__init__.py', 'harness.py']
✅ agent stub ready


In [21]:
import sys, json, os, requests
sys.path.insert(0, "/content"); sys.path.insert(0, "/content/eval")
from tasks import TASKS
from harness import score_task

# Confirm Tavily is really working
key = os.environ.get("TAVILY_API_KEY","")
r = requests.post("https://api.tavily.com/search",
    json={"api_key": key, "query": "test", "max_results": 1}, timeout=15)
print("Tavily check:", r.status_code)

with open("/content/e4_url.txt") as f: E4_URL = f.read().strip()
print("URL:", E4_URL)

def calc(a):
    e = a.get("expression","").replace("^","**")
    if not all(c in set("0123456789+-*/(). ") for c in e): raise ValueError(e)
    return eval(e, {"__builtins__":{}}, {})
def wc(a): return len(a.get("text","").split())
def search(a):
    r = requests.post("https://api.tavily.com/search",
        json={"api_key": key, "query": a.get("query",""), "max_results":3}, timeout=20)
    r.raise_for_status()
    return [{"title":x.get("title"),"url":x.get("url"),"snippet":x.get("content")} for x in r.json().get("results",[])]
def weather(a):
    g = requests.get("https://geocoding-api.open-meteo.com/v1/search",
        params={"name":a.get("city",""),"count":1}, timeout=15).json()
    if not g.get("results"): raise ValueError("city")
    lat,lon = g["results"][0]["latitude"], g["results"][0]["longitude"]
    return requests.get("https://api.open-meteo.com/v1/forecast",
        params={"latitude":lat,"longitude":lon,"current_weather":True}, timeout=15).json().get("current_weather",{})
TOOLS = {"calculator":calc,"word_count":wc,"web_search":search,"get_weather":weather}

class TC:
    def __init__(s,n,a): s.tool_name,s.arguments,s.is_error=n,a,False
class Tr:
    def __init__(s):
        s.tool_calls=[]; s.tool_results=[]; s.tool_names_called=set()
        s.had_tool_error=False; s.hit_max_steps=False; s.final_answer=""; s.steps=0

SCHEMA = [
    {"type":"function","function":{"name":"calculator","description":"Evaluate arithmetic","parameters":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}},
    {"type":"function","function":{"name":"word_count","description":"Count words","parameters":{"type":"object","properties":{"text":{"type":"string"}},"required":["text"]}}},
    {"type":"function","function":{"name":"web_search","description":"Search the web","parameters":{"type":"object","properties":{"query":{"type":"string"}},"required":["query"]}}},
    {"type":"function","function":{"name":"get_weather","description":"Get weather","parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}},
]
SYS = "You are a helpful AI agent with access to tools: calculator, word_count, web_search, and get_weather. Use tools when you need real information or exact computation. Respond directly when you already know the answer."

results = []
for i, task in enumerate(TASKS, 1):
    msgs = [{"role":"system","content":SYS},{"role":"user","content":task.prompt}]
    tr = Tr(); final=""; hit=False
    for step in range(3):
        tr.steps = step+1
        try:
            out = requests.post(f"{E4_URL}/chat",
                json={"messages":msgs,"tools":SCHEMA}, timeout=180).json()
        except Exception as e:
            final = f"ERR:{e}"; break
        if out.get("tool_calls"):
            am = {"role":"assistant","content":None,"tool_calls":[
                {"type":"function","function":{"name":tc["name"],"arguments":json.dumps(tc.get("arguments",{}))}}
                for tc in out["tool_calls"]]}
            msgs.append(am)
            for tc in out["tool_calls"]:
                n = tc["name"]; a = tc.get("arguments",{})
                stc = TC(n,a)
                try: c = json.dumps({"result": TOOLS[n](a)})
                except Exception as e:
                    stc.is_error=True; tr.had_tool_error=True; c = json.dumps({"error":str(e)})
                tr.tool_names_called.add(n); tr.tool_calls.append(stc)
                msgs.append({"role":"tool","content":c})
        else:
            final = out.get("text","") or ""; break
    else: hit=True
    tr.final_answer=final; tr.hit_max_steps=hit
    try:
        r = score_task(task, tr)
        passed = getattr(r,"passed",None)
        if passed is None:
            fr = getattr(r,"failure_reasons",None)
            passed = (len(fr)==0) if fr is not None else None
    except Exception as e:
        passed=None
    results.append({"task_id":task.id,"passed":bool(passed) if passed is not None else None})
    print(f"[{i:2d}/23] {'✓' if passed else ('?' if passed is None else '✗')} {task.id}")

n = sum(1 for r in results if r["passed"])
print(f"\n{'='*60}\nE4 RESULT: {n}/23  (V1 + fixed parser + REAL Tavily)\n{'='*60}")
with open("/content/eval_results_e4.json","w") as f:
    json.dump({"condition":"E4","adapter":"agentlab_qwen_lora_7b (V1)",
               "parser":"v2_two_pass","tavily":"real","passed":n,"total":23,
               "results":results}, f, indent=2)
print("Saved.")

Tavily check: 200
URL: https://ae06-34-142-230-136.ngrok-free.app
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "128 * 47"}}
</tool_call>
[PARSED] {'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '128 * 47'}}], 'text': None}
INFO:     34.142.230.136:0 - "POST /chat HTTP/1.1" 200 OK
[RAW] 128 * 47 = 6016.
[PARSED] {'tool_calls': [], 'text': '128 * 47 = 6016.'}
INFO:     34.142.230.136:0 - "POST /chat HTTP/1.1" 200 OK
[ 1/23] ✓ calc_basic_1
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "900 / 12"}}
</tool_call>
[PARSED] {'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '900 / 12'}}], 'text': None}
INFO:     34.142.230.136:0 - "POST /chat HTTP/1.1" 200 OK
[RAW] 900 divided by 12 is 75.
[PARSED] {'tool_calls': [], 'text': '900 divided by 12 is 75.'}
INFO:     34.142.230.136:0 - "POST /chat HTTP/1.1" 200 OK
[ 2/23] ✓ calc_basic_2
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "(15 + 5) * 3

In [22]:
from google.colab import files
files.download("/content/eval_results_e4.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import sys, types, os

# Make sure the eval package path works
os.makedirs("/content/eval", exist_ok=True)
open("/content/eval/__init__.py", "a").close()

# Create a stub "agent" module so harness.py's imports succeed
fake_agent = types.ModuleType("agent")

class Agent:
    pass

class RunTrace:
    def __init__(self, *args, **kwargs):
        # Accept anything harness.py passes; store as attributes
        for k, v in kwargs.items():
            setattr(self, k, v)
        # Sensible defaults so nothing downstream breaks
        if not hasattr(self, "tool_calls"):
            self.tool_calls = []
        if not hasattr(self, "tool_results"):
            self.tool_results = []
        if not hasattr(self, "steps"):
            self.steps = 0
        if not hasattr(self, "final_answer"):
            self.final_answer = ""
        if not hasattr(self, "hit_max_steps"):
            self.hit_max_steps = False

class ToolCall:
    def __init__(self, name=None, arguments=None, **kwargs):
        self.name = name
        self.arguments = arguments or {}

class ToolResult:
    def __init__(self, name=None, output=None, error=None, **kwargs):
        self.name = name
        self.output = output
        self.error = error

fake_agent.Agent = Agent
fake_agent.RunTrace = RunTrace
fake_agent.ToolCall = ToolCall
fake_agent.ToolResult = ToolResult

sys.modules["agent"] = fake_agent

# Also expose them as top-level names in case harness does `from agent import *`
globals()["Agent"] = Agent
globals()["RunTrace"] = RunTrace
globals()["ToolCall"] = ToolCall
globals()["ToolResult"] = ToolResult

# Sanity check
import importlib
importlib.reload(sys.modules.get("agent"))
print("✅ Stub 'agent' module installed")
print("Now re-run Cell 11.")

ModuleNotFoundError: spec not found for the module 'agent'

In [19]:
import sys, types, os

os.makedirs("/content/eval", exist_ok=True)
open("/content/eval/__init__.py", "a").close()

fake_agent = types.ModuleType("agent")

class Agent:
    pass

class RunTrace:
    def __init__(self, *args, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
        self.tool_calls = getattr(self, "tool_calls", [])
        self.tool_results = getattr(self, "tool_results", [])
        self.steps = getattr(self, "steps", 0)
        self.final_answer = getattr(self, "final_answer", "")
        self.hit_max_steps = getattr(self, "hit_max_steps", False)

class ToolCall:
    def __init__(self, name=None, arguments=None, **kwargs):
        self.name = name
        self.arguments = arguments or {}

class ToolResult:
    def __init__(self, name=None, output=None, error=None, **kwargs):
        self.name = name
        self.output = output
        self.error = error

fake_agent.Agent = Agent
fake_agent.RunTrace = RunTrace
fake_agent.ToolCall = ToolCall
fake_agent.ToolResult = ToolResult

sys.modules["agent"] = fake_agent

print("✅ Stub 'agent' installed. Now re-run Cell 11.")

✅ Stub 'agent' installed. Now re-run Cell 11.


In [17]:
import sys, json, requests
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

# Adjust these imports if your file/function names differ
from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

print("Running E4 against:", E4_URL)
print("Tasks loaded:", len(TASKS))
print("-" * 50)

SYSTEM_PROMPT = (
    "You are a helpful AI agent with access to tools: calculator, word_count, "
    "web_search, and get_weather. Use tools when you need real information or "
    "exact computation. Respond directly when you already know the answer."
)

results = []
for i, task in enumerate(TASKS, 1):
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task["prompt"]},
        ],
        "tools": task["tools"],
    }
    try:
        resp = requests.post(f"{E4_URL}/chat", json=payload, timeout=180)
        result = resp.json()
    except Exception as e:
        result = {"tool_calls": [], "text": f"ERROR: {e}"}

    passed = score_task(task, result, max_steps=3)
    results.append({"task_id": task.get("id", f"task_{i}"), "passed": passed})
    print(f"[{i:2d}/23] {'✓' if passed else '✗'} {task.get('id', '?')}")

n_passed = sum(1 for r in results if r["passed"])

print("\n" + "=" * 60)
print(f"E4 RESULT: {n_passed}/23  (V1 adapter + fixed parser)")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({
        "condition": "E4",
        "adapter": "agentlab_qwen_lora_7b (V1)",
        "parser": "v2_two_pass",
        "passed": n_passed,
        "total": 23,
        "results": results,
    }, f, indent=2)

print("\nSaved /content/eval_results_e4.json")

ModuleNotFoundError: No module named 'agent'

In [20]:
import sys, json, requests
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

print("Running E4 against:", E4_URL)
print("Tasks loaded:", len(TASKS))
print("-" * 50)

SYSTEM_PROMPT = (
    "You are a helpful AI agent with access to tools: calculator, word_count, "
    "web_search, and get_weather. Use tools when you need real information or "
    "exact computation. Respond directly when you already know the answer."
)

results = []
for i, task in enumerate(TASKS, 1):
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task["prompt"]},
        ],
        "tools": task["tools"],
    }
    try:
        resp = requests.post(f"{E4_URL}/chat", json=payload, timeout=180)
        result = resp.json()
    except Exception as e:
        result = {"tool_calls": [], "text": f"ERROR: {e}"}

    passed = score_task(task, result, max_steps=3)
    results.append({"task_id": task.get("id", f"task_{i}"), "passed": passed})
    print(f"[{i:2d}/23] {'✓' if passed else '✗'} {task.get('id', '?')}")

n_passed = sum(1 for r in results if r["passed"])

print("\n" + "=" * 60)
print(f"E4 RESULT: {n_passed}/23  (V1 adapter + fixed parser)")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({
        "condition": "E4",
        "adapter": "agentlab_qwen_lora_7b (V1)",
        "parser": "v2_two_pass",
        "passed": n_passed,
        "total": 23,
        "results": results,
    }, f, indent=2)

print("\nSaved /content/eval_results_e4.json")

Running E4 against: https://883c-34-125-110-47.ngrok-free.app
Tasks loaded: 23
--------------------------------------------------


TypeError: 'Task' object is not subscriptable

In [21]:
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")
from tasks import TASKS, Task

t = TASKS[0]
print("Type:", type(t))
print("All attributes (dir):")
print([a for a in dir(t) if not a.startswith("_")])

print("\n--- Attribute values ---")
for a in [a for a in dir(t) if not a.startswith("_")]:
    try:
        v = getattr(t, a)
        if callable(v):
            continue
        if isinstance(v, (list, dict)) and len(str(v)) > 200:
            print(f"{a}: <{type(v).__name__} len={len(v)}>")
        else:
            print(f"{a}: {v!r}")
    except Exception as e:
        print(f"{a}: ERROR {e}")

print("\n--- Task.__init__ signature ---")
import inspect
print(inspect.signature(Task.__init__))

Type: <class 'tasks.Task'>
All attributes (dir):
['answer_contains', 'answer_contains_any', 'category', 'expect_tool_error', 'id', 'numeric_check', 'prompt', 'required_tools']

--- Attribute values ---
answer_contains: []
answer_contains_any: []
category: 'math'
expect_tool_error: False
id: 'calc_basic_1'
numeric_check: (6016, 0.5)
prompt: 'What is 128 * 47?'
required_tools: {'calculator'}

--- Task.__init__ signature ---
(self, id: 'str', prompt: 'str', required_tools: 'set[str]' = <factory>, answer_contains: 'list[str]' = <factory>, answer_contains_any: 'list[str]' = <factory>, numeric_check: 'tuple[float, float] | None' = None, expect_tool_error: 'bool' = False, category: 'str' = 'general') -> None


In [22]:
import sys, json, requests
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

# Helper: read attribute OR dict key from a Task object
def get(t, key, default=None):
    if isinstance(t, dict):
        return t.get(key, default)
    return getattr(t, key, default)

# Figure out the right key names
sample = TASKS[0]
print("Sample task type:", type(sample).__name__)
print("Sample attribute names:", [a for a in dir(sample) if not a.startswith("_")])

# Try to auto-detect
PROMPT_KEYS  = ["prompt", "question", "user_prompt", "input", "query"]
TOOLS_KEYS   = ["tools", "tool_schema", "tool_definitions", "available_tools"]
ID_KEYS      = ["id", "task_id", "name", "task_name"]

def pick(obj, keys, default=None):
    for k in keys:
        v = get(obj, k, None)
        if v is not None:
            return k, v
    return None, default

pk, pv = pick(sample, PROMPT_KEYS)
tk, tv = pick(sample, TOOLS_KEYS)
ik, iv = pick(sample, ID_KEYS)

print(f"\nDetected prompt key: {pk}")
print(f"Detected tools key:  {tk}")
print(f"Detected id key:     {ik}")

if pk is None:
    raise SystemExit("Could not find a prompt attribute. Paste the diagnostic output to me.")
if tk is None:
    raise SystemExit("Could not find a tools attribute. Paste the diagnostic output to me.")

print("\n" + "-" * 50)

SYSTEM_PROMPT = (
    "You are a helpful AI agent with access to tools: calculator, word_count, "
    "web_search, and get_weather. Use tools when you need real information or "
    "exact computation. Respond directly when you already know the answer."
)

results = []
for i, task in enumerate(TASKS, 1):
    prompt = get(task, pk)
    tools  = get(task, tk)
    tid    = get(task, ik, f"task_{i}") if ik else f"task_{i}"

    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        "tools": tools,
    }
    try:
        resp = requests.post(f"{E4_URL}/chat", json=payload, timeout=180)
        result = resp.json()
    except Exception as e:
        result = {"tool_calls": [], "text": f"ERROR: {e}"}

    try:
        passed = score_task(task, result, max_steps=3)
    except TypeError:
        # try a couple of alternate signatures
        try:
            passed = score_task(task, result)
        except TypeError:
            passed = bool(result.get("tool_calls"))  # last-resort fallback

    results.append({"task_id": str(tid), "passed": bool(passed)})
    print(f"[{i:2d}/23] {'✓' if passed else '✗'} {tid}")

n_passed = sum(1 for r in results if r["passed"])

print("\n" + "=" * 60)
print(f"E4 RESULT: {n_passed}/23  (V1 adapter + fixed parser)")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({
        "condition": "E4",
        "adapter": "agentlab_qwen_lora_7b (V1)",
        "parser": "v2_two_pass",
        "passed": n_passed,
        "total": 23,
        "results": results,
    }, f, indent=2)

print("\nSaved /content/eval_results_e4.json")

Sample task type: Task
Sample attribute names: ['answer_contains', 'answer_contains_any', 'category', 'expect_tool_error', 'id', 'numeric_check', 'prompt', 'required_tools']

Detected prompt key: prompt
Detected tools key:  None
Detected id key:     id


SystemExit: Could not find a tools attribute. Paste the diagnostic output to me.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [23]:
import sys, inspect
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")
from harness import score_task
print("score_task signature:", inspect.signature(score_task))
print("---- source (first 1500 chars) ----")
print(inspect.getsource(score_task)[:1500])

score_task signature: (task: 'Task', trace: 'RunTrace') -> 'TaskResult'
---- source (first 1500 chars) ----
def score_task(task: Task, trace: RunTrace) -> TaskResult:
    """Apply every check the task defines. A task fails if ANY check fails;
    we collect ALL failing reasons (not just the first) so a report can
    show every problem in a single run, not just the first one hit."""
    failure_reasons: list[str] = []

    # --- Check 1: required tools were actually called ---
    missing_tools = task.required_tools - trace.tool_names_called
    if missing_tools:
        failure_reasons.append(f"missing_required_tools:{sorted(missing_tools)}")

    # --- Check 2: tool-level errors ---
    # Normally a tool error is a failure. But for tasks that deliberately
    # trigger one (e.g. division by zero) and want to see how the agent
    # HANDLES it, expect_tool_error flips this: no error occurring is the
    # failure, not the other way around.
    if task.expect_tool_error:
        if not

In [24]:
import sys, json, requests
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

# ---- Tool schema (MUST match colab_train_7b.py exactly) ----
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression, e.g. '12 * (3 + 4)'.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "A math expression to evaluate"}},
            "required": ["expression"]},
    }},
    {"type": "function", "function": {
        "name": "word_count",
        "description": "Count the number of words in a piece of text.",
        "parameters": {"type": "object", "properties": {
            "text": {"type": "string", "description": "The text to count words in"}},
            "required": ["text"]},
    }},
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the live web for current information. Returns structured JSON "
                       "results with title, url, snippet, and source for each hit.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "The search query"}},
            "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current real-time weather for a city (temperature in Celsius, condition).",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name"}},
            "required": ["city"]},
    }},
]

SYSTEM_PROMPT = (
    "You are a helpful AI agent with access to tools: calculator, word_count, "
    "web_search, and get_weather. Use tools when you need real information or "
    "exact computation. Respond directly when you already know the answer."
)

print("Running E4 against:", E4_URL)
print("Tasks loaded:", len(TASKS))
print("-" * 60)

results = []
for i, task in enumerate(TASKS, 1):
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": task.prompt},
        ],
        "tools": TOOLS_SCHEMA,
    }
    try:
        resp = requests.post(f"{E4_URL}/chat", json=payload, timeout=180)
        result = resp.json()
    except Exception as e:
        result = {"tool_calls": [], "text": f"ERROR: {e}"}

    # Try score_task with max_steps first, then without
    try:
        passed = score_task(task, result, max_steps=3)
    except TypeError:
        try:
            passed = score_task(task, result)
        except TypeError:
            passed = None  # do NOT silently substitute a fake score

    if passed is None:
        print(f"[{i:2d}/23] ?? {task.id}  <-- score_task signature mismatch, see Cell 11.0b output")
        results.append({"task_id": task.id, "passed": None, "raw_result": result})
        continue

    results.append({"task_id": task.id, "passed": bool(passed)})
    print(f"[{i:2d}/23] {'✓' if passed else '✗'} {task.id}")

scored = [r for r in results if r["passed"] is not None]
n_passed = sum(1 for r in scored if r["passed"])
n_scored = len(scored)

print("\n" + "=" * 60)
if n_scored == 23:
    print(f"E4 RESULT: {n_passed}/23  (V1 adapter + fixed parser)")
else:
    print(f"E4 PARTIAL: {n_passed}/{n_scored} scored (of 23 attempted)")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({
        "condition": "E4",
        "adapter": "agentlab_qwen_lora_7b (V1)",
        "parser": "v2_two_pass",
        "passed": n_passed,
        "scored": n_scored,
        "total": 23,
        "results": results,
    }, f, indent=2, default=str)

print("\nSaved /content/eval_results_e4.json")

Running E4 against: https://883c-34-125-110-47.ngrok-free.app
Tasks loaded: 23
------------------------------------------------------------


AttributeError: 'dict' object has no attribute 'tool_names_called'

In [25]:
import os
os.environ["TAVILY_API_KEY"] = "PASTE_YOUR_TAVILY_KEY_HERE"   # or skip
print("TAVILY_API_KEY set:", bool(os.environ.get("TAVILY_API_KEY")))

TAVILY_API_KEY set: True


In [26]:
import sys, json, os, requests
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()

# ============================================================
# Local tool executors (mirror mcp_server.py)
# ============================================================
def exec_calculator(args):
    expr = args.get("expression", "")
    allowed = set("0123456789+-*/(). ")
    expr2 = expr.replace("^", "**")
    if not all(c in allowed for c in expr2):
        raise ValueError(f"unsafe expression: {expr}")
    return eval(expr2, {"__builtins__": {}}, {})

def exec_word_count(args):
    return len(args.get("text", "").split())

def exec_web_search(args):
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        raise RuntimeError("No TAVILY_API_KEY set")
    r = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": key, "query": args.get("query", ""), "max_results": 3},
        timeout=20,
    )
    r.raise_for_status()
    data = r.json()
    return [{"title": x.get("title"), "url": x.get("url"), "snippet": x.get("content")}
            for x in data.get("results", [])]

def exec_get_weather(args):
    city = args.get("city", "")
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}, timeout=15,
    ).json()
    if not geo.get("results"):
        raise ValueError(f"City not found: {city}")
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=15,
    ).json()
    return wx.get("current_weather", {})

TOOL_EXECUTORS = {
    "calculator": exec_calculator,
    "word_count": exec_word_count,
    "web_search": exec_web_search,
    "get_weather": exec_get_weather,
}

# ============================================================
# RunTrace-compatible stub
# ============================================================
class SimpleToolCall:
    def __init__(self, name, args):
        self.tool_name = name
        self.arguments = args
        self.result = None
        self.is_error = False

class SimpleTrace:
    def __init__(self):
        self.tool_calls = []
        self.tool_results = []
        self.tool_names_called = set()
        self.had_tool_error = False
        self.hit_max_steps = False
        self.final_answer = ""
        self.steps = 0

# ============================================================
# Tool schema (must match training)
# ============================================================
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression, e.g. '12 * (3 + 4)'.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "A math expression to evaluate"}},
            "required": ["expression"]},
    }},
    {"type": "function", "function": {
        "name": "word_count",
        "description": "Count the number of words in a piece of text.",
        "parameters": {"type": "object", "properties": {
            "text": {"type": "string", "description": "The text to count words in"}},
            "required": ["text"]},
    }},
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the live web for current information. Returns structured JSON "
                       "results with title, url, snippet, and source for each hit.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "The search query"}},
            "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current real-time weather for a city (temperature in Celsius, condition).",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name"}},
            "required": ["city"]},
    }},
]

SYSTEM_PROMPT = (
    "You are a helpful AI agent with access to tools: calculator, word_count, "
    "web_search, and get_weather. Use tools when you need real information or "
    "exact computation. Respond directly when you already know the answer."
)

MAX_STEPS = 3

# ============================================================
# Sanity check
# ============================================================
h = requests.get(f"{E4_URL}/health", timeout=15)
print("Health:", h.status_code, h.json())
print("Tasks loaded:", len(TASKS))
print("-" * 60)

# ============================================================
# Main loop
# ============================================================
results = []

for i, task in enumerate(TASKS, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": task.prompt},
    ]

    trace = SimpleTrace()
    final_text = ""
    hit_max = False

    for step in range(MAX_STEPS):
        trace.steps = step + 1
        payload = {"messages": messages, "tools": TOOLS_SCHEMA}

        try:
            resp = requests.post(f"{E4_URL}/chat", json=payload, timeout=180)
            out = resp.json()
        except Exception as e:
            final_text = f"ERROR: {e}"
            break

        if out.get("tool_calls"):
            # Append assistant message with tool_calls (OpenAI-style)
            assistant_msg = {"role": "assistant", "content": None, "tool_calls": []}
            for tc in out["tool_calls"]:
                assistant_msg["tool_calls"].append({
                    "type": "function",
                    "function": {
                        "name": tc["name"],
                        "arguments": json.dumps(tc.get("arguments", {})),
                    },
                })
            messages.append(assistant_msg)

            # Execute each tool call locally
            for tc in out["tool_calls"]:
                name = tc["name"]
                args = tc.get("arguments", {})
                stc = SimpleToolCall(name, args)
                try:
                    result = TOOL_EXECUTORS[name](args)
                    stc.result = result
                    content = json.dumps({"result": result})
                except Exception as e:
                    stc.result = str(e)
                    stc.is_error = True
                    trace.had_tool_error = True
                    content = json.dumps({"error": str(e)})

                trace.tool_names_called.add(name)
                trace.tool_calls.append(stc)

                messages.append({"role": "tool", "content": content})
        else:
            final_text = out.get("text", "") or ""
            break
    else:
        hit_max = True

    trace.final_answer = final_text
    trace.hit_max_steps = hit_max

    # Score
    try:
        task_result = score_task(task, trace)
        # Try common TaskResult attribute names
        passed = getattr(task_result, "passed", None)
        if passed is None:
            passed = getattr(task_result, "success", None)
        if passed is None:
            failures = getattr(task_result, "failure_reasons", None)
            if failures is not None:
                passed = (len(failures) == 0)
            else:
                passed = None
    except Exception as e:
        passed = None
        print(f"      score_task error: {e}")

    results.append({
        "task_id": task.id,
        "passed": bool(passed) if passed is not None else None,
    })
    mark = "✓" if passed else ("?" if passed is None else "✗")
    print(f"[{i:2d}/23] {mark} {task.id}")

n = sum(1 for r in results if r["passed"])
n_scored = sum(1 for r in results if r["passed"] is not None)

print("\n" + "=" * 60)
print(f"E4 RESULT: {n}/23  (V1 adapter + fixed parser)")
if n_scored < 23:
    print(f"  WARNING: only {n_scored}/23 tasks were scorable")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({
        "condition": "E4",
        "adapter": "agentlab_qwen_lora_7b (V1)",
        "parser": "v2_two_pass",
        "passed": n,
        "scored": n_scored,
        "total": 23,
        "results": results,
    }, f, indent=2)

print("\nSaved /content/eval_results_e4.json")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [27]:
print("model loaded:", "model" in globals())
print("tokenizer loaded:", "tokenizer" in globals())
print("app defined:", "app" in globals())

import requests
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print("LOCAL server:", r.status_code, r.text)
except Exception as e:
    print("LOCAL server dead:", e)

model loaded: True
tokenizer loaded: True
app defined: True
INFO:     127.0.0.1:60016 - "GET /health HTTP/1.1" 200 OK
LOCAL server: 200 {"status":"ok","adapter":"/content/content/agentlab_qwen_lora_7b","parser":"v2_two_pass"}


In [28]:
import threading, uvicorn, nest_asyncio, os
from pyngrok import ngrok
from fastapi import FastAPI, Request
import re, json

# Rebuild app if needed
if "app" not in globals():
    app = FastAPI()
    TOOL_CALL_CLOSED = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
    TOOL_CALL_OPEN   = re.compile(r"<tool_call>\s*(\{.*?\})", re.DOTALL)

    def parse_model_output(raw_text: str) -> dict:
        matches = TOOL_CALL_CLOSED.findall(raw_text)
        if matches:
            calls = []
            for m in matches:
                try:
                    p = json.loads(m)
                    if p.get("name"):
                        calls.append({"name": p["name"], "arguments": p.get("arguments", {})})
                except json.JSONDecodeError:
                    continue
            if calls:
                return {"tool_calls": calls, "text": None}
        m = TOOL_CALL_OPEN.search(raw_text)
        if m:
            try:
                p = json.loads(m.group(1))
                if p.get("name"):
                    return {"tool_calls": [{"name": p["name"], "arguments": p.get("arguments", {})}], "text": None}
            except json.JSONDecodeError:
                pass
        return {"tool_calls": [], "text": raw_text.strip()}

    @app.post("/chat")
    async def chat(request: Request):
        body = await request.json()
        messages = body.get("messages", [])
        tools = body.get("tools", [])
        prompt = tokenizer.apply_chat_template(
            messages, tools=tools if tools else None,
            add_generation_prompt=True, tokenize=False,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=512,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return parse_model_output(raw)

    @app.get("/health")
    async def health():
        return {"status": "ok", "parser": "v2_two_pass"}

    print("app rebuilt")

# Kill any dead ngrok tunnels
ngrok.kill()

NGROK_AUTH_TOKEN = "30x34MIoplFkIzcMdvm1Ti8cB7T_3Dp9L4kX2JozPrZQZMHvV"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

public_url = ngrok.connect(8000)
E4_URL = public_url.public_url

with open("/content/e4_url.txt", "w") as f:
    f.write(E4_URL)

# Make sure server thread is running
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

print("\n" + "=" * 60)
print("NEW E4 SERVER URL:", E4_URL)
print("=" * 60)


NEW E4 SERVER URL: https://47d6-34-125-110-47.ngrok-free.app


INFO:     Started server process [1826]
INFO:     Waiting for application startup.


In [29]:
from google.colab import files
files.download("/content/eval_results_e4.json")

FileNotFoundError: Cannot find file: /content/eval_results_e4.json

In [30]:
import sys, json, os, requests, time
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/eval")

from tasks import TASKS
from harness import score_task

with open("/content/e4_url.txt") as f:
    E4_URL = f.read().strip()
print("Using URL:", E4_URL)

h = requests.get(f"{E4_URL}/health", timeout=15)
print("Health:", h.status_code, h.text[:200])
print("-" * 60)

# --- local tool executors ---
def exec_calculator(args):
    expr = args.get("expression", "")
    allowed = set("0123456789+-*/(). ")
    expr2 = expr.replace("^", "**")
    if not all(c in allowed for c in expr2):
        raise ValueError(f"unsafe: {expr}")
    return eval(expr2, {"__builtins__": {}}, {})

def exec_word_count(args):
    return len(args.get("text", "").split())

def exec_web_search(args):
    key = os.environ.get("TAVILY_API_KEY")
    if not key or key == "PASTE_YOUR_TAVILY_KEY_HERE":
        raise RuntimeError("no tavily key")
    r = requests.post("https://api.tavily.com/search",
        json={"api_key": key, "query": args.get("query",""), "max_results": 3}, timeout=20)
    r.raise_for_status()
    return [{"title": x.get("title"), "url": x.get("url"), "snippet": x.get("content")}
            for x in r.json().get("results", [])]

def exec_get_weather(args):
    city = args.get("city","")
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=15).json()
    if not geo.get("results"):
        raise ValueError(f"city not found: {city}")
    lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]
    wx = requests.get("https://api.open-meteo.com/v1/forecast",
                      params={"latitude": lat, "longitude": lon, "current_weather": True},
                      timeout=15).json()
    return wx.get("current_weather", {})

TOOL_EXECUTORS = {
    "calculator": exec_calculator,
    "word_count": exec_word_count,
    "web_search": exec_web_search,
    "get_weather": exec_get_weather,
}

class SimpleToolCall:
    def __init__(self, name, args):
        self.tool_name = name
        self.arguments = args
        self.is_error = False

class SimpleTrace:
    def __init__(self):
        self.tool_calls = []
        self.tool_results = []
        self.tool_names_called = set()
        self.had_tool_error = False
        self.hit_max_steps = False
        self.final_answer = ""
        self.steps = 0

TOOLS_SCHEMA = [
    {"type":"function","function":{"name":"calculator","description":"Evaluate a basic arithmetic expression, e.g. '12 * (3 + 4)'.","parameters":{"type":"object","properties":{"expression":{"type":"string","description":"A math expression to evaluate"}},"required":["expression"]}}},
    {"type":"function","function":{"name":"word_count","description":"Count the number of words in a piece of text.","parameters":{"type":"object","properties":{"text":{"type":"string","description":"The text to count words in"}},"required":["text"]}}},
    {"type":"function","function":{"name":"web_search","description":"Search the live web for current information.","parameters":{"type":"object","properties":{"query":{"type":"string","description":"The search query"}},"required":["query"]}}},
    {"type":"function","function":{"name":"get_weather","description":"Get current weather for a city.","parameters":{"type":"object","properties":{"city":{"type":"string","description":"City name"}},"required":["city"]}}},
]

SYSTEM_PROMPT = ("You are a helpful AI agent with access to tools: calculator, word_count, "
                 "web_search, and get_weather. Use tools when you need real information or "
                 "exact computation. Respond directly when you already know the answer.")

MAX_STEPS = 3
results = []

for i, task in enumerate(TASKS, 1):
    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":task.prompt},
    ]
    trace = SimpleTrace()
    final_text = ""
    hit_max = False

    for step in range(MAX_STEPS):
        trace.steps = step + 1
        try:
            resp = requests.post(f"{E4_URL}/chat",
                                 json={"messages": messages, "tools": TOOLS_SCHEMA},
                                 timeout=180)
            out = resp.json()
        except Exception as e:
            final_text = f"ERROR: {e}"
            break

        if out.get("tool_calls"):
            assistant_msg = {"role":"assistant","content":None,"tool_calls":[]}
            for tc in out["tool_calls"]:
                assistant_msg["tool_calls"].append({
                    "type":"function",
                    "function":{"name":tc["name"],"arguments":json.dumps(tc.get("arguments",{}))}
                })
            messages.append(assistant_msg)

            for tc in out["tool_calls"]:
                name = tc["name"]
                args = tc.get("arguments", {})
                stc = SimpleToolCall(name, args)
                try:
                    res = TOOL_EXECUTORS[name](args)
                    content = json.dumps({"result": res})
                except Exception as e:
                    stc.is_error = True
                    trace.had_tool_error = True
                    content = json.dumps({"error": str(e)})
                trace.tool_names_called.add(name)
                trace.tool_calls.append(stc)
                messages.append({"role":"tool","content":content})
        else:
            final_text = out.get("text", "") or ""
            break
    else:
        hit_max = True

    trace.final_answer = final_text
    trace.hit_max_steps = hit_max

    try:
        task_result = score_task(task, trace)
        passed = getattr(task_result, "passed", None)
        if passed is None:
            passed = getattr(task_result, "success", None)
        if passed is None:
            fr = getattr(task_result, "failure_reasons", None)
            passed = (len(fr) == 0) if fr is not None else None
    except Exception as e:
        passed = None
        print(f"      score error: {e}")

    results.append({"task_id": task.id, "passed": bool(passed) if passed is not None else None})
    mark = "✓" if passed else ("?" if passed is None else "✗")
    print(f"[{i:2d}/23] {mark} {task.id}")

n = sum(1 for r in results if r["passed"])
n_scored = sum(1 for r in results if r["passed"] is not None)

print("\n" + "=" * 60)
print(f"E4 RESULT: {n}/23  (V1 adapter + fixed parser)")
if n_scored < 23:
    print(f"  WARNING: only {n_scored}/23 scorable")
print("=" * 60)

with open("/content/eval_results_e4.json", "w") as f:
    json.dump({"condition":"E4","adapter":"agentlab_qwen_lora_7b (V1)",
               "parser":"v2_two_pass","passed":n,"scored":n_scored,
               "total":23,"results":results}, f, indent=2)

print("\nSaved. Now run the download cell:")
print('from google.colab import files; files.download("/content/eval_results_e4.json")')

Using URL: https://47d6-34-125-110-47.ngrok-free.app
INFO:     34.125.110.47:0 - "GET /health HTTP/1.1" 200 OK
Health: 200 {"status":"ok","adapter":"/content/content/agentlab_qwen_lora_7b","parser":"v2_two_pass"}
------------------------------------------------------------
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "128 * 47"}}
</tool_call>
[PARSED] {'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '128 * 47'}}], 'text': None}
INFO:     34.125.110.47:0 - "POST /chat HTTP/1.1" 200 OK
[RAW] 128 * 47 = 6016.
[PARSED] {'tool_calls': [], 'text': '128 * 47 = 6016.'}
INFO:     34.125.110.47:0 - "POST /chat HTTP/1.1" 200 OK
[ 1/23] ✓ calc_basic_1
[RAW] <tool_call>
{"name": "calculator", "arguments": {"expression": "900 / 12"}}
</tool_call>
[PARSED] {'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '900 / 12'}}], 'text': None}
INFO:     34.125.110.47:0 - "POST /chat HTTP/1.1" 200 OK
[RAW] 900 divided by 12 is 75.
[PARSED] {'tool_calls'

In [32]:
import os
os.environ["TAVILY_API_KEY"] = "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"
print("Set:", os.environ["TAVILY_API_KEY"][:10], "...")

Set: tvly-dev-2 ...


In [33]:
from google.colab import files
files.download("/content/eval_results_e4.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import os
os.environ["TAVILY_API_KEY"] = "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"  # paste the real one
print("Set:", os.environ["TAVILY_API_KEY"][:12])

Set: tvly-dev-DtA


In [35]:
import os
os.environ["TAVILY_API_KEY"] = "tvly-YOUR-REAL-KEY-HERE"  # paste the real one
print("Set:", os.environ["TAVILY_API_KEY"][:12])

Set: tvly-YOUR-RE


In [36]:
!cp /content/eval_results_e4.json /content/eval_results_e4_NOTAVILY.json
from google.colab import files
files.download("/content/eval_results_e4_NOTAVILY.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os
os.environ["TAVILY_API_KEY"] = "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"
print("Key set:", os.environ["TAVILY_API_KEY"][:16], "...")

# Quick test — should return results, not an error
import requests
r = requests.post("https://api.tavily.com/search",
    json={"api_key": os.environ["TAVILY_API_KEY"],
          "query": "what year was the first iPhone released",
          "max_results": 2}, timeout=20)
print("Status:", r.status_code)
data = r.json()
if "results" in data:
    print("✅ Tavily works. First result:", data["results"][0]["title"])
else:
    print("❌ Something wrong:", data)

Key set: tvly-dev-DtArZ-6 ...
Status: 200
✅ Tavily works. First result: History of the iPhone


In [2]:
!cp /content/eval_results_e4.json /content/eval_results_e4_TAVILY.json
from google.colab import files
files.download("/content/eval_results_e4_TAVILY.json")

cp: cannot stat '/content/eval_results_e4.json': No such file or directory


FileNotFoundError: Cannot find file: /content/eval_results_e4_TAVILY.json